# Etapa 2 — Extracción, limpieza y transformación de datos
### Reto Global Classroom — Analítica de Airbnb | Ciudad: Austin, TX
**Rol:** Ingeniería de datos
**Fuente de datos:** Inside Airbnb — `listings.csv` (Austin, TX)

Este cuaderno documenta el análisis exploratorio inicial y el proceso de limpieza, transformación y preparación del conjunto de datos de Airbnb para Austin, de cara a las etapas posteriores del proyecto (análisis exploratorio, modelos de regresión/clasificación y tablero).

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_columns", 50)

RAW_PATH = "listings.csv"
df = pd.read_csv(RAW_PATH, low_memory=False)
df.shape

(11111, 90)

## 1. Análisis exploratorio de los datos iniciales
Número de variables, tipos de dato y proporción de nulos por columna.

In [2]:
print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])
print()
print("Tipos de dato:")
print(df.dtypes.value_counts())

Número de filas: 11111
Número de columnas: 90

Tipos de dato:
float64    37
str        28
int64      25
Name: count, dtype: int64


In [3]:
null_pct = (df.isnull().mean() * 100).round(1).sort_values(ascending=False)
null_pct[null_pct > 0].to_frame("% nulos")

,% nulos
neighborhood_overview,100.0
host_since,100.0
host_response_time,100.0
host_thumbnail_url,100.0
host_acceptance_rate,100.0
host_response_rate,100.0
host_verifications,100.0
neighbourhood,100.0
host_total_listings_count,100.0
host_neighbourhood,100.0


In [4]:
print("Filas totalmente duplicadas:", df.duplicated().sum())
print("IDs de anuncio duplicados:", df["id"].duplicated().sum())

Filas totalmente duplicadas: 0
IDs de anuncio duplicados: 0


In [5]:
# "faltantes disfrazados": valores de texto que en realidad significan vacío pero no son NaN
placeholders = ["N", "NA", "N/A", "n/a", "null", "NULL", "None", "none", "-", "--"]
obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
placeholders_encontrados = {}
for c in obj_cols:
    hits = [v for v in df[c].dropna().unique() if str(v).strip() in placeholders]
    if hits:
        placeholders_encontrados[c] = hits

print("Columnas con placeholders de 'faltante' disfrazados de texto:",
      placeholders_encontrados if placeholders_encontrados else "ninguna")

Columnas con placeholders de 'faltante' disfrazados de texto: {'host_about': ['-']}


**Hallazgos del análisis inicial:**
- El dataset trae 90 columnas y 11.111 anuncios de Austin, sin filas ni IDs duplicados.
- Se revisaron las columnas de texto en busca de "faltantes disfrazados" (`"N"`, `"NA"`, `"-"`, etc. que no son `NaN` pero significan vacío); solo apareció un caso aislado en `host_about`, columna que de todas formas se descarta por ser texto libre sin procesar.
- 14 columnas llegan 100% vacías (p. ej. `host_since`, `host_response_rate`, `bathrooms`, `instant_bookable`, `neighbourhood`), es decir, el campo existe pero nunca se diligenció en esta extracción.
- Varias columnas de identificación y metadatos de scraping (URLs, `scrape_id`, `source`, cotizaciones puntuales de precio) no aportan valor analítico para responder las preguntas de negocio.
- Los puntajes de reseñas (`review_scores_*`) y `reviews_per_month` tienen 16.7% de nulos, que corresponden exactamente a los anuncios sin reseñas todavía (`number_of_reviews == 0`), no a datos faltantes por error.
- `price` (variable objetivo para el modelo de regresión) tiene 10.0% de nulos.

## 2. Selección de variables potencialmente relevantes
Se eliminan: (a) columnas 100% vacías, y (b) columnas técnicas/identificadoras (URLs, IDs internos de scraping, cotizaciones puntuales, `estimated_revenue_l365d` por ser una combinación derivada de precio y ocupación que generaría fuga de información hacia el modelo de precio).

In [6]:
cols_100_null = df.columns[df.isnull().mean() == 1.0].tolist()

cols_tech = [
    "listing_url", "scrape_id", "picture_url", "host_url", "host_profile_id",
    "host_profile_url", "host_picture_url", "calendar_last_scraped",
    "source", "host_id", "host_name",
    "price_quote_checkin_date", "price_quote_checkout_date", "price_quote_raw",
    "price_quote_total_price", "price_quote_price_per_night",
    "estimated_revenue_l365d",
]
cols_tech = [c for c in cols_tech if c in df.columns]
# nota: last_scraped, first_review y last_review no se eliminan aun -> se usan mas abajo
# para calcular la recencia de las reseñas antes de descartar las fechas crudas

print("Columnas 100% nulas eliminadas:", len(cols_100_null))
print("Columnas técnicas/identificadoras eliminadas:", len(cols_tech))

df = df.drop(columns=list(set(cols_100_null + cols_tech)))
df.shape

Columnas 100% nulas eliminadas: 14
Columnas técnicas/identificadoras eliminadas: 17


(11111, 59)

## 3. Corrección de formatos
`price` llega como texto con símbolo de moneda (`"$1,599.40"`); `bathrooms_text` es una descripción libre (`"2.5 baths"`, `"1 shared bath"`); varias booleanas llegan como `'t'`/`'f'`; `host_location` mezcla ciudad y estado en texto libre; la antigüedad del host está repartida en 4 columnas de años/meses; y `last_scraped`, `first_review` y `last_review` llegan como texto (`"2026-09-19"`) en vez de fecha.

In [7]:
# price: texto -> numérico
df["price"] = (df["price"].astype(str)
               .str.replace(r"[\$,]", "", regex=True)
               .replace("nan", np.nan)
               .astype(float))

# bathrooms_text -> bathrooms (numérico) + bathrooms_shared (bandera)
def parse_bathrooms(text):
    if pd.isna(text):
        return np.nan
    if "half" in text.lower():
        return 0.5
    m = re.search(r"[\d.]+", text)
    return float(m.group()) if m else np.nan

df["bathrooms"] = df["bathrooms_text"].apply(parse_bathrooms)
df["bathrooms_shared"] = df["bathrooms_text"].fillna("").str.contains("shared", case=False).astype(int)

# booleanas 't'/'f' -> 1/0
for c in ["host_is_superhost", "host_has_profile_pic", "host_identity_verified", "has_availability"]:
    df[c] = df[c].map({"t": 1, "f": 0})

# host_location -> estado del host + si es local (Texas)
df["host_state"] = df["host_location"].str.extract(r",\s*([A-Z]{2})$")
df["host_state"] = df["host_state"].fillna("OTHER")
df["host_is_local"] = (df["host_state"] == "TX").astype(int)
freq_state = df["host_state"].value_counts(normalize=True)
rare_state = freq_state[freq_state < 0.01].index
df["host_state"] = df["host_state"].where(~df["host_state"].isin(rare_state), "OTHER")

# antigüedad del host en meses (a partir de las columnas año/mes ya separadas)
df["user_tenure_months"] = df["hosts_time_as_user_years"] * 12 + df["hosts_time_as_user_months"]
df["host_tenure_months"] = df["hosts_time_as_host_years"] * 12 + df["hosts_time_as_host_months"]

# fechas: texto -> datetime, y de ahi a una magnitud numerica (dias)
df["last_scraped"] = pd.to_datetime(df["last_scraped"])
df["first_review"] = pd.to_datetime(df["first_review"])
df["last_review"] = pd.to_datetime(df["last_review"])

df["review_recency_days"] = (df["last_scraped"] - df["last_review"]).dt.days
df["listing_tenure_days"] = (df["last_scraped"] - df["first_review"]).dt.days
# sin reseñas -> no hay fecha de reseña; se marca con un valor alto ("inactivo"),
# distinguible gracias a la bandera has_reviews que se crea mas adelante
df["review_recency_days"] = df["review_recency_days"].fillna(9999)
df["listing_tenure_days"] = df["listing_tenure_days"].fillna(9999)

df[["price", "bathrooms", "bathrooms_shared", "host_state", "host_is_local",
    "host_tenure_months", "review_recency_days", "listing_tenure_days"]].head()

,price,bathrooms,bathrooms_shared,host_state,host_is_local,host_tenure_months,review_recency_days,listing_tenure_days
0,79.74,2.5,0,TX,1,46,9999.0,9999.0
1,158.00,1.0,0,OTHER,0,0,2.0,2.0
2,NaN,1.0,0,TX,1,44,9999.0,9999.0
3,428.80,1.0,0,OTHER,0,79,9999.0,9999.0
4,499.00,1.0,0,TX,1,67,1120.0,1597.0
